# Notebook 04: Transfer Learning with ResNet-18 and EfficientNet-B0

**Objective:** Explore, fine-tune, and compare two dominant transfer-learning paradigms for industrial defect classification.

### Core Concepts Explored:
1. **Residual Learning (ResNet-18):** How identity skip connections solve vanishing gradients.
2. **Compound Scaling & Depthwise Convolutions (EfficientNet-B0):** Scaling depth, width, and resolution simultaneously for maximal parameter efficiency.
3. **Two-Stage Fine-Tuning Strategy:** Why freezing early layers preserves generic low-level edge detectors while tuning deep semantic heads.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.classification.resnet_classifier import ResNetClassifier
from src.models.classification.efficientnet_classifier import EfficientNetClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Architectural Parameter Comparison

Let's compare model complexity, parameter counts, and freezing behaviors.

In [ ]:
resnet = ResNetClassifier(num_classes=6, pretrained=True).to(device)
enet = EfficientNetClassifier(num_classes=6, pretrained=True).to(device)

resnet.freeze_backbone()
enet.freeze_backbone()

res_params = resnet.get_num_parameters()
enet_params = enet.get_num_parameters()

comparison = pd.DataFrame([
    {
        "Model": "ResNet-18",
        "Total Params": f"{res_params['total_parameters']:,}",
        "Trainable (Frozen Stage 1)": f"{res_params['trainable_parameters']:,}",
        "Frozen Params": f"{res_params['frozen_parameters']:,}",
        "Key Innovation": "Residual Skip Connections (y = F(x) + x)"
    },
    {
        "Model": "EfficientNet-B0",
        "Total Params": f"{enet_params['total_parameters']:,}",
        "Trainable (Frozen Stage 1)": f"{enet_params['trainable_parameters']:,}",
        "Frozen Params": f"{enet_params['frozen_parameters']:,}",
        "Key Innovation": "Compound Scaling + Depthwise Separable Convs"
    }
])

display(comparison)

## 2. Why ResNet Skip Connections Prevent Vanishing Gradients

In a plain deep network, the gradient of the loss with respect to early layer $x$ is a chain of matrix multiplications:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \prod_{k=l}^{L-1} W_k$$
If weights $|W_k| < 1$, gradients decay exponentially to zero (vanishing gradient).

In ResNet, with $x_{l+1} = \mathcal{F}(x_l) + x_l$:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_{l+1}} \left( \frac{\partial \mathcal{F}}{\partial x_l} + 1 \right)$$
The constant **$+1$** term guarantees that gradient signals flow directly across hundreds of layers without attenuation!

In [ ]:
# Smoke test forward pass on both architectures
x = torch.randn(2, 3, 224, 224).to(device)
out_res = resnet(x)
out_enet = enet(x)

print(f"ResNet-18 Output Shape:     {out_res.shape}")
print(f"EfficientNet-B0 Output Shape: {out_enet.shape}")